Loading the cleaned data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/nassau_clean.csv")

features = ["Product Name", "Factory", "Region", "Ship Mode", "Distance KM", "Units", "Sales"]
target = "Lead Time Clean"

X = df[features]
y = df[target]

print(X.shape, y.shape)
X.head()

(10194, 7) (10194,)


,Product Name,Factory,Region,Ship Mode,Distance KM,Units,Sales
0,Wonka Bar - Milk Chocolate,Wicked Choccy's,Interior,Standard Class,1563.508797,2,6.50
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Interior,Standard Class,1160.082944,2,7.50
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Interior,Standard Class,2188.317712,3,10.47
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Interior,Standard Class,2188.317712,3,10.80
4,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Atlantic,Standard Class,1008.219033,3,11.25


Split the data and prepare the inputs

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat_cols = ["Product Name", "Factory", "Region", "Ship Mode"]
num_cols = ["Distance KM", "Units", "Sales"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

print(X_train.shape, X_test.shape)

(8155, 7) (2039, 7)


Training the three models and comparing them

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    })
    fitted[name] = pipe

pd.DataFrame(results).round(3)

,Model,MAE,RMSE,R2
0,Linear Regression,0.926,1.106,0.621
1,Random Forest,0.937,1.166,0.578
2,Gradient Boosting,0.919,1.103,0.623


In [4]:
import joblib

joblib.dump(fitted["Gradient Boosting"], "../models/gradient_boosting.pkl")
pd.DataFrame(results).round(3).to_csv("../reports/model_results.csv", index=False)
print("saved")

saved


Simulating every product at every factory

In [5]:
import joblib

model = joblib.load("../models/gradient_boosting.pkl")

factory_coords = {
    "Lot's O' Nuts": (32.881893, -111.768036),
    "Wicked Choccy's": (32.076176, -81.088371),
    "Sugar Shack": (48.11914, -96.18115),
    "Secret Factory": (41.446333, -90.565487),
    "The Other Factory": (35.1175, -89.971107),
}

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

cols = []
for fac, (lat, lon) in factory_coords.items():
    sim = df[features].copy()
    sim["Factory"] = fac
    sim["Distance KM"] = haversine(lat, lon, df["Dest Lat"], df["Dest Lon"])
    sim["Predicted"] = model.predict(sim[features])
    cols.append(sim.groupby("Product Name")["Predicted"].mean().rename(fac))

sim_table = pd.concat(cols, axis=1).round(3)
sim_table

,Lot's O' Nuts,Wicked Choccy's,Sugar Shack,Secret Factory,The Other Factory
Product Name,,,,,
Everlasting Gobstopper,4.116,4.192,4.144,4.151,4.155
Fizzy Lifting Drinks,4.411,4.410,4.424,4.401,4.435
Fun Dip,4.073,4.044,3.911,4.032,4.107
Hair Toffee,3.408,3.374,3.479,3.442,3.441
Kazookles,4.129,4.151,4.182,4.153,4.159
Laffy Taffy,4.452,4.363,4.416,4.356,4.363
Lickable Wallpaper,4.085,4.094,4.154,4.123,4.131
Nerds,4.948,5.089,4.879,5.089,5.041
SweeTARTS,5.168,5.167,5.184,5.175,5.171


Building the recommendation table

In [6]:
current = df.groupby("Product Name")["Factory"].first()

rows = []
for p in sim_table.index:
    cur = current[p]
    cur_pred = sim_table.loc[p, cur]
    best = sim_table.loc[p].idxmin()
    best_pred = sim_table.loc[p, best]
    rows.append({
        "Product": p,
        "Current Factory": cur,
        "Current Lead Time": cur_pred,
        "Best Factory": best,
        "Best Lead Time": best_pred,
        "Reduction %": (cur_pred - best_pred) / cur_pred * 100,
    })

rec = pd.DataFrame(rows).sort_values("Reduction %", ascending=False).round(2)
rec

,Product,Current Factory,Current Lead Time,Best Factory,Best Lead Time,Reduction %
3,Hair Toffee,The Other Factory,3.44,Wicked Choccy's,3.37,1.95
5,Laffy Taffy,Sugar Shack,4.42,Secret Factory,4.36,1.36
14,Wonka Gum,Secret Factory,4.03,Lot's O' Nuts,3.98,1.31
6,Lickable Wallpaper,Secret Factory,4.12,Lot's O' Nuts,4.08,0.92
0,Everlasting Gobstopper,Secret Factory,4.15,Lot's O' Nuts,4.12,0.84
4,Kazookles,The Other Factory,4.16,Lot's O' Nuts,4.13,0.72
1,Fizzy Lifting Drinks,Sugar Shack,4.42,Secret Factory,4.40,0.52
13,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,4.27,Wicked Choccy's,4.25,0.33
8,SweeTARTS,Sugar Shack,5.18,Wicked Choccy's,5.17,0.33
12,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,4.32,Lot's O' Nuts,4.30,0.32


Add distance change and profit to the recommendations

In [8]:
def avg_dist(product, fac):
    lat, lon = factory_coords[fac]
    sub = df[df["Product Name"] == product]
    return haversine(lat, lon, sub["Dest Lat"], sub["Dest Lon"]).mean()

rec["Current Dist KM"] = [avg_dist(p, f) for p, f in zip(rec["Product"], rec["Current Factory"])]
rec["Best Dist KM"] = [avg_dist(p, f) for p, f in zip(rec["Product"], rec["Best Factory"])]
rec["Distance Change %"] = (rec["Best Dist KM"] - rec["Current Dist KM"]) / rec["Current Dist KM"] * 100

prod_info = df.groupby("Product Name").agg(Total_Profit=("Gross Profit", "sum"), Orders=("Row ID", "count"))
rec["Total Profit"] = rec["Product"].map(prod_info["Total_Profit"])
rec["Orders"] = rec["Product"].map(prod_info["Orders"])

rec.round(1)

,Product,Current Factory,Current Lead Time,Best Factory,Best Lead Time,Reduction %,Current Dist KM,Best Dist KM,Distance Change %,Total Profit,Orders
3,Hair Toffee,The Other Factory,3.4,Wicked Choccy's,3.4,2.0,1781.7,2022.0,13.5,59.5,4
5,Laffy Taffy,Sugar Shack,4.4,Secret Factory,4.4,1.4,1865.8,1372.0,-26.5,33.5,10
14,Wonka Gum,Secret Factory,4.0,Lot's O' Nuts,4.0,1.3,1331.3,2275.1,70.9,310.7,120
6,Lickable Wallpaper,Secret Factory,4.1,Lot's O' Nuts,4.1,0.9,1487.9,2161.8,45.3,3930.0,94
0,Everlasting Gobstopper,Secret Factory,4.2,Lot's O' Nuts,4.1,0.8,1437.8,1728.6,20.2,104.0,3
4,Kazookles,The Other Factory,4.2,Lot's O' Nuts,4.1,0.7,1603.0,2258.1,40.9,92.8,96
1,Fizzy Lifting Drinks,Sugar Shack,4.4,Secret Factory,4.4,0.5,1700.1,1149.2,-32.4,47.2,6
13,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,4.3,Wicked Choccy's,4.2,0.3,2112.6,1878.7,-11.1,19357.5,2064
8,SweeTARTS,Sugar Shack,5.2,Wicked Choccy's,5.2,0.3,1800.6,1527.8,-15.1,28.7,10
12,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,4.3,Lot's O' Nuts,4.3,0.3,1891.7,2078.8,9.9,18610.2,2015


Adding a risk verdict to each recommendation

In [9]:
def verdict(r):
    if r["Reduction %"] == 0:
        return "Keep current"
    if r["Distance Change %"] > 5:
        return "High risk: farther distance"
    if r["Orders"] < 20:
        return "Low confidence: few orders"
    return "Recommend"

rec["Verdict"] = rec.apply(verdict, axis=1)
rec[["Product", "Current Factory", "Best Factory", "Reduction %", "Distance Change %", "Orders", "Verdict"]].round(1)

,Product,Current Factory,Best Factory,Reduction %,Distance Change %,Orders,Verdict
3,Hair Toffee,The Other Factory,Wicked Choccy's,2.0,13.5,4,High risk: farther distance
5,Laffy Taffy,Sugar Shack,Secret Factory,1.4,-26.5,10,Low confidence: few orders
14,Wonka Gum,Secret Factory,Lot's O' Nuts,1.3,70.9,120,High risk: farther distance
6,Lickable Wallpaper,Secret Factory,Lot's O' Nuts,0.9,45.3,94,High risk: farther distance
0,Everlasting Gobstopper,Secret Factory,Lot's O' Nuts,0.8,20.2,3,High risk: farther distance
4,Kazookles,The Other Factory,Lot's O' Nuts,0.7,40.9,96,High risk: farther distance
1,Fizzy Lifting Drinks,Sugar Shack,Secret Factory,0.5,-32.4,6,Low confidence: few orders
13,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Wicked Choccy's,0.3,-11.1,2064,Recommend
8,SweeTARTS,Sugar Shack,Wicked Choccy's,0.3,-15.1,10,Low confidence: few orders
12,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Lot's O' Nuts,0.3,9.9,2015,High risk: farther distance


Save the simulation and recommendation tables

In [10]:
sim_table.to_csv("../data/processed/simulation_table.csv")
rec.to_csv("../data/processed/recommendations.csv", index=False)
print("saved")

saved
